# Part 20: Archaeology — Repository Sync

Repository synchronization is the process of bringing a local copy of a remote repository up to
date. It uses MST diff operations (Part 11) and CAR archives (Part 9) to efficiently transfer only
the blocks that changed.

This notebook models a simplified sync engine that compares local and remote MST roots and applies
the diff.

**What you'll learn:**

- Sync engine architecture
- Local vs remote MST comparison
- Block transfer via CAR files
- Conflict detection and resolution

**Prerequisites:** Parts 9, 11, 16, 19

**Estimated Time:** 20-25 minutes

## Step 1: Sync Engine Architecture

The sync engine compares the local MST root CID with the remote root CID. If they differ, it
requests a diff from the remote, which contains the operations needed to bring the local tree up to
date.

In [ ]:
// Sync engine — compares local and remote state
@interface SyncEngine : NSObject
@property (nonatomic, strong) NSString *localRoot;
@property (nonatomic, strong) NSString *remoteRoot;
- (BOOL)needsSync;
- (NSString *)status;
@end

@implementation SyncEngine
- (BOOL)needsSync {
    return ![_localRoot isEqualToString:_remoteRoot];
}
- (NSString *)status {
    if ([self needsSync]) {
        return @"OUT_OF_SYNC";
    }
    return @"IN_SYNC";
}
@end

SyncEngine *engine = [[SyncEngine alloc] init];
engine.localRoot = @"bafyrei-abc";
engine.remoteRoot = @"bafyrei-xyz";
NSLog(@"Status: %@", [engine status]);
NSLog(@"Needs sync: %d", [engine needsSync]);

## Step 2: Diff Operations

A diff contains add, update, and delete operations. Each operation references a collection and rkey
path.

In [ ]:
// Diff operation model
@interface DiffOp : NSObject
@property (nonatomic, strong) NSString *type;
@property (nonatomic, strong) NSString *collection;
@property (nonatomic, strong) NSString *rkey;
@property (nonatomic, strong) NSString *cid;
- (NSString *)description;
@end

@implementation DiffOp
- (NSString *)description {
    return [NSString stringWithFormat:@"%@ %@/%@ -> %@", _type, _collection, _rkey, _cid];
}
@end

// Create diff operations
DiffOp *add = [[DiffOp alloc] init];
add.type = @"add";
add.collection = @"app.bsky.feed.post";
add.rkey = @"3k2la";
add.cid = @"bafyrei-new";

DiffOp *del = [[DiffOp alloc] init];
del.type = @"delete";
del.collection = @"app.bsky.feed.post";
del.rkey = @"3k1old";
del.cid = @"bafyrei-old";

NSLog(@"%@", [add description]);
NSLog(@"%@", [del description]);

## Step 3: Applying a Diff

The sync engine applies diff operations to the local store. Add operations insert new records,
delete operations remove them.

In [ ]:
// Local store that can apply diffs
@interface LocalStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *records;
- (void)applyDiff:(DiffOp *)op;
- (int)recordCount;
@end

@implementation LocalStore
- (instancetype)init {
    self = [super init];
    if (self) { _records = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)applyDiff:(DiffOp *)op {
    NSString *key = [NSString stringWithFormat:@"%@/%@", [op collection], [op rkey]];
    if ([[op type] isEqualToString:@"add"] || [[op type] isEqualToString:@"update"]) {
        [_records setObject:[op cid] forKey:key];
        NSLog(@"Applied %@: %@", [op type], key);
    } else if ([[op type] isEqualToString:@"delete"]) {
        [_records removeObjectForKey:key];
        NSLog(@"Applied delete: %@", key);
    }
}
- (int)recordCount {
    return [_records count];
}
@end

LocalStore *store = [[LocalStore alloc] init];

// Apply add
DiffOp *addOp = [[DiffOp alloc] init];
addOp.type = @"add";
addOp.collection = @"app.bsky.feed.post";
addOp.rkey = @"3k2la";
addOp.cid = @"bafyrei-new";
[store applyDiff:addOp];

// Apply delete
DiffOp *delOp = [[DiffOp alloc] init];
delOp.type = @"delete";
delOp.collection = @"app.bsky.feed.post";
delOp.rkey = @"3k2la";
delOp.cid = @"bafyrei-new";
[store applyDiff:delOp];

NSLog(@"Records: %d", [store recordCount]);

## Step 4: Production Anchor

In Garazyk, repository sync is in `Sources/Repository/`:

- `RepoSync.h` — orchestrates getRepo and subscribeRepos
- `MSTDiff.h` — computes MST-level diff operations
- `CARReader.h` — reads CAR archives for block transfer

The production sync engine handles concurrent writes during sync, CAR block verification, and MST
proof validation.

## Exercise

Add a `batchApply:` method to `LocalStore` that takes an NSArray of DiffOp objects and applies them
all. Return the number of operations applied.

In [ ]:
// Exercise: implement batchApply:
// Your code here...

## Solution

In [ ]:
// Solution: batchApply iterates and applies each diff
@interface BatchStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *records;
- (void)applyDiff:(DiffOp *)op;
- (int)batchApply:(NSArray *)ops;
@end

@implementation BatchStore
- (instancetype)init {
    self = [super init];
    if (self) { _records = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)applyDiff:(DiffOp *)op {
    NSString *key = [NSString stringWithFormat:@"%@/%@", [op collection], [op rkey]];
    if ([[op type] isEqualToString:@"add"]) {
        [_records setObject:[op cid] forKey:key];
    } else if ([[op type] isEqualToString:@"delete"]) {
        [_records removeObjectForKey:key];
    }
}
- (int)batchApply:(NSArray *)ops {
    int count = 0;
    for (int i = 0; i < [ops count]; i++) {
        DiffOp *op = [ops objectAtIndex:i];
        [self applyDiff:op];
        count++;
    }
    return count;
}
@end

BatchStore *bs = [[BatchStore alloc] init];
DiffOp *op1 = [[DiffOp alloc] init];
op1.type = @"add";
op1.collection = @"app.bsky.feed.post";
op1.rkey = @"a";
op1.cid = @"cid1";
DiffOp *op2 = [[DiffOp alloc] init];
op2.type = @"add";
op2.collection = @"app.bsky.feed.like";
op2.rkey = @"b";
op2.cid = @"cid2";
int applied = [bs batchApply:@[op1, op2]];
NSLog(@"Applied %d ops, %d records", applied, [[bs records] count]);

## What to Read Next

You've completed the archaeology series. The full curriculum covers:

- Parts 0-5: Objective-C foundations
- Parts 6-11: ATProto data primitives
- Parts 12-17: Service construction
- Parts 18-20: Archaeology and production mapping

For production code, explore the `Garazyk/Sources/` directory.